### Visualización, preprocesamiento y modelo base

Antes de usar modelos preentrenados, necesitamos un **punto de referencia** (*baseline*): un modelo clásico de NLP con el que comparar. Así podremos medir cuánto aporta realmente el transfer learning.

Este notebook está estructurado como sigue:

- **BLOQUE A**: carga de los datos.
- **BLOQUE B**: visualización.
- **BLOQUE C**: preprocesamiento del texto.
- **BLOQUE D**: partición en entrenamiento y validación (80/20).
- **BLOQUE E**: vectorización del texto con TF-IDF.
- **BLOQUE F**: entrenamiento del modelo base (regresión logística).
- **BLOQUE G**: evaluación sobre las noticias nuevas (2020-21).
- **BLOQUE H**: exportación.

In [ ]:
# Preparación del entorno: funciona tanto en Google Colab como en local
import os, sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB and not os.path.exists("seminario-manadine-transfer-learning"):
    !git clone -q https://github.com/Vins16/seminario-manadine-transfer-learning.git
    %pip install -q -r seminario-manadine-transfer-learning/requirements.txt

if EN_COLAB:
    RAIZ = "seminario-manadine-transfer-learning"
else:
    RAIZ = "." if os.path.exists("data") else ".."
sys.path.insert(0, RAIZ)
DATA = f"{RAIZ}/data"
MODELS = f"{RAIZ}/models"

In [ ]:
import re
import unicodedata

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from stop_words import get_stop_words
from wordcloud import WordCloud

sns.set_theme(style="whitegrid")

### BLOQUE A: carga de los datos

In [ ]:
df = pd.???(f"{DATA}/noticias_2018.csv")
df_nuevas = pd.???(f"{DATA}/noticias_2020_21.csv")
df.head()

### BLOQUE B: visualización

In [ ]:
# ¿Cuántas noticias hay de cada clase? (1 = falsa, 0 = verdadera)
sns.???(data=df, x="target")
plt.title("Distribución de la variable objetivo")
plt.show()

In [ ]:
# Número de noticias falsas y verdaderas por tema
pd.???(df["Topic"], df["target"]).plot(kind="barh", figsize=(8, 5))
plt.title("Noticias por tema y clase")
plt.legend(["Verdadera (0)", "Falsa (1)"])
plt.show()

In [ ]:
# Distribución de la longitud de las noticias según su clase (eje X en escala logarítmica)
sns.???(data=df, x="longitud", hue="target", bins=40, log_scale=True)
plt.title("Longitud de las noticias (caracteres)")
plt.show()

¿Recuerdas la función `proporcion_mayusculas` del notebook 1? Comprobemos si los bulos "gritan" más.

In [ ]:
def proporcion_mayusculas(texto):
    letras = [c for c in texto if c.isalpha()]
    if len(letras) == 0:
        return 0
    return sum(c.isupper() for c in letras) / len(letras)


df["mayusculas"] = df["texto"].???

sns.???(data=df, x="target", y="mayusculas", showfliers=False)
plt.title("Proporción de mayúsculas por clase")
plt.show()

### BLOQUE C: preprocesamiento del texto

Los modelos clásicos tratan cada palabra como un símbolo independiente: para ellos, *"Gobierno"*, *"gobierno"* y *"gobiernó"* son tres palabras distintas. Por eso normalizamos el texto:

1. Pasamos todo a minúsculas.
2. Quitamos tildes.
3. Eliminamos números y signos de puntuación.
4. Eliminamos las **palabras vacías** (*stopwords*): "el", "de", "que"... que aparecen en todas las noticias y no aportan información.

In [ ]:
def quitar_tildes(texto):
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(c for c in texto if not unicodedata.combining(c))


STOPWORDS_ES = {quitar_tildes(p) for p in get_stop_words("spanish")}
print(len(STOPWORDS_ES), "palabras vacías. Algunas:", sorted(STOPWORDS_ES)[:15])

In [ ]:
def preprocesar(texto):
    texto = quitar_tildes(texto.???)
    texto = re.sub(r"[^a-z\s]", " ", texto)    # nos quedamos solo con letras y espacios
    tokens = [t for t in texto.split() if t ??? STOPWORDS_ES and len(t) > 2]
    return " ".join(tokens)


ejemplo = df["texto"].iloc[0][:250]
print("ANTES: ", ejemplo)
print("\nDESPUÉS:", preprocesar(ejemplo))

> Estas funciones también están en el fichero `utils.py`, para poder reutilizarlas en el notebook 4.

In [ ]:
df["texto_prep"] = df["texto"].???
df_nuevas["texto_prep"] = df_nuevas["texto"].???

¿Qué palabras son más frecuentes en cada clase? Una **nube de palabras** lo muestra de un vistazo: cuanto más grande, más frecuente.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(16, 6))
for eje, clase, titulo in zip(ejes, [1, 0], ["Noticias falsas", "Noticias verdaderas"]):
    texto_clase = " ".join(df.loc[df["target"] == clase, "texto_prep"])
    nube = WordCloud(width=800, height=500, background_color="white", max_words=80).???(texto_clase)
    eje.imshow(nube)
    eje.set_title(titulo, fontsize=16)
    eje.axis("off")
plt.show()

### BLOQUE D: partición del conjunto de datos en entrenamiento y validación (80/20)

Reservamos un 20 % de las noticias de 2018 para **validar** el modelo con datos que no ha visto durante el entrenamiento. Usamos `stratify` para que ambas partes mantengan la proporción de noticias falsas y verdaderas.

Partimos los **índices** de las filas (y no directamente los textos) para poder reutilizar exactamente la misma partición en el notebook 4.

In [ ]:
idx_train, idx_val = ???(df.index, test_size=???, stratify=df["target"], random_state=42)

df["particion"] = "train"
df.loc[idx_val, "particion"] = "val"
df["particion"].value_counts()

In [ ]:
X_train, y_train = df.loc[idx_train, "texto_prep"], df.loc[idx_train, "target"]
X_val, y_val = df.loc[idx_val, "texto_prep"], df.loc[idx_val, "target"]

### BLOQUE E: vectorización del texto con TF-IDF

Los modelos solo entienden números. **TF-IDF** (*Term Frequency - Inverse Document Frequency*) convierte cada noticia en un vector donde cada posición corresponde a una palabra del vocabulario. El peso de cada palabra es mayor cuanto más aparece en esa noticia y menor cuanto más común es en el resto de noticias.

Usaremos también **bigramas** (parejas de palabras consecutivas, como *"ultima hora"*).

In [ ]:
vectorizador = TfidfVectorizer(max_features=???, ngram_range=???, min_df=2)

X_train_vec = vectorizador.???(X_train)
X_val_vec = vectorizador.???(X_val)

print(X_train_vec.shape, X_val_vec.shape)

**¿Por qué usamos `fit_transform` en entrenamiento y solo `transform` en validación?**

El vocabulario y los pesos IDF deben aprenderse **solo** con los datos de entrenamiento. Si los aprendiéramos también con los de validación, estaríamos dejando que el modelo "espiara" información que en la realidad no tendría.

In [ ]:
# Algunas palabras del vocabulario aprendido
vocabulario = vectorizador.get_feature_names_out()
print(vocabulario[1000:1030])

### BLOQUE F: entrenamiento del modelo base (regresión logística)

La regresión logística estima la probabilidad de que una noticia sea falsa a partir de una combinación lineal de sus variables (aquí, los pesos TF-IDF de cada palabra). Es sencilla, rápida e interpretable: un clásico difícil de batir con pocos datos.

Más información sobre este modelo [aquí](https://cienciadedatos.net/documentos/py17-regresion-logistica-python.html).

In [ ]:
modelo_lr = ???(max_iter=1000, C=10)
modelo_lr.???(X_train_vec, y_train)

In [ ]:
# Resultados sobre el conjunto de validación
pred_val = modelo_lr.???(X_val_vec)

print(f"Accuracy: {accuracy_score(???):.2f}\n")
print(classification_report(???, target_names=["verdadera", "falsa"]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, pred_val, display_labels=["verdadera", "falsa"], cmap="Blues")
plt.title("Matriz de confusión (validación)")
plt.show()

Una ventaja de la regresión logística es que podemos ver **qué palabras empujan hacia cada clase** mirando sus coeficientes.

In [ ]:
coeficientes = pd.Series(modelo_lr.???[0], index=vocabulario).sort_values()

fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
coeficientes.tail(15).plot(kind="barh", ax=ejes[0], color="tab:red", title="Palabras que indican noticia FALSA")
coeficientes.head(15).plot(kind="barh", ax=ejes[1], color="tab:blue", title="Palabras que indican noticia VERDADERA")
plt.tight_layout()
plt.show()

### BLOQUE G: evaluación sobre las noticias nuevas (2020-21)

Llega la prueba de fuego: noticias publicadas **dos o tres años después** y sobre temas que el modelo nunca ha visto.

In [ ]:
X_nuevas_vec = vectorizador.???(df_nuevas["texto_prep"])
pred_nuevas = modelo_lr.predict(X_nuevas_vec)

print(f"Accuracy: {accuracy_score(df_nuevas['target'], pred_nuevas):.2f}\n")
print(classification_report(df_nuevas["target"], pred_nuevas, target_names=["verdadera", "falsa"]))

In [ ]:
# ¿Conoce el modelo la palabra "covid"?
for palabra in ["covid", "vacuna", "pandemia", "mascarilla", "gobierno"]:
    print(f"{palabra:<12} {'sí' if palabra ??? vectorizador.vocabulary_ else 'no'}")

El rendimiento cae claramente. Un modelo basado en palabras solo conoce el vocabulario que vio al entrenar: *"covid"*, *"pandemia"* o *"mascarilla"* son invisibles para él, aunque aparezcan en casi la mitad de las noticias nuevas.

Este problema se conoce como **cambio de dominio** (*domain shift*), y es justo donde el **transfer learning** puede ayudarnos: un modelo preentrenado con millones de textos ya "sabe español" antes de ver nuestros datos. Lo comprobaremos en el notebook 4.

### BLOQUE H: exportación

Guardamos la partición para reutilizarla en el notebook 4 y el modelo base (vectorizador + regresión logística) en un único `Pipeline`.

In [ ]:
df[["Id", "Topic", "texto", "longitud", "target", "particion"]].???(f"{DATA}/noticias_2018_particion.csv", index=False)

baseline = Pipeline([("tfidf", vectorizador), ("modelo", modelo_lr)])
joblib.???(baseline, f"{MODELS}/baseline_tfidf_lr.joblib")